# DGEBA / 4,4′-DDS: Dual-mode Tg Bayesian Optimizer

Real mode never uses literature values or synthetic demonstrations as MD labels. The current team DDS pilot reached ~8% cure but provides no validated Tg observations. The published 100:50 starting structure is NOT an available cured-network optimization observation.

In [ ]:
from pathlib import Path
import pandas as pd
from src.dds_optimizer import build_composition_pool, read_csv, propose, REQUIRED_OBS, REQUIRED_CAND
ROOT=Path.cwd()
obs=read_csv(ROOT / "data/md_tg_observations.csv", REQUIRED_OBS)
print("Approved MD Tg observations presently:", len(obs))

## Mode A — fixed composition, choose an existing, PhysCheck-approved cured conversion state

The real manifest currently has no validated DDS Tg-ready networks. Returning no proposal is the **correct** result. It is not a runtime error.

In [ ]:
networks=read_csv(ROOT/"data/dds_network_manifest.csv", REQUIRED_CAND)
conversion_proposals, conversion_info=propose(networks, obs, "conversion", tg_min=150, tg_max=220)
print(conversion_info)
conversion_proposals

## Mode B — propose compositions and desired conversions

The candidate pool is a planning set, not observed Tg data. Proposals require creation, curing, equilibration and validation before Tg MD.

In [ ]:
compositions=build_composition_pool(n_dgeba=100,dds_counts=[40,45,50,55,60],conversions=[60,75,85,90])
composition_proposals, composition_info=propose(compositions, obs, "composition", tg_min=150, tg_max=220)
print(composition_info)
composition_proposals[["candidate_id","n_DGEBA","n_DDS","ratio","conversion_pct","proposed_job_type","selection_method"]]

## Optional: **SYNTHETIC ONLY** GP smoke test

These are invented software-testing fixtures with no measured Tg or valid LAMMPS topology. They cannot be used in the research dataset.

In [ ]:
demo_dir=ROOT/"examples/DEMO_SYNTHETIC_ONLY"
synthetic_pool=read_csv(demo_dir/"DEMO_candidates.csv", REQUIRED_CAND)
synthetic_obs=read_csv(demo_dir/"DEMO_observations.csv", REQUIRED_OBS)
synthetic_proposals, synthetic_info=propose(synthetic_pool, synthetic_obs, "conversion", allow_synthetic_demo=True)
print("**SYNTHETIC SOFTWARE TEST — NOT MD RESULTS**", synthetic_info["method"])
synthetic_proposals[["candidate_id","conversion_pct","predicted_tg_C","p_tg_in_window"]]

## Integration note

Only real network files that pass the entire PhysicsCheck (connectivity, force field, charges, achieved conversion and NPT equilibration) can be marked `CURED_NETWORK_VALIDATED`. The `generate_and_cure_network` proposals require Experiment/Curing Agent execution. BO observations require reviewed apparent Tg fits and protocol provenance. See README.md.